In [1]:
import numpy as np
import joblib
import os
import pandas as pd
from scipy.stats import kurtosis, skew

In [2]:
def get_features(acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, interval=6):
    acc_x_linear = acc_x - np.mean(acc_x)
    acc_y_linear = acc_y - np.mean(acc_y)
    acc_z_linear = acc_z - np.mean(acc_z)
    linear_acc_magnitude = np.sqrt(acc_x_linear**2 + acc_y_linear**2 + acc_z_linear**2)

    gyro_magnitude = np.sqrt(gyro_x**2 + gyro_y**2 + gyro_z**2)

    features = []

    for start in range(0, len(linear_acc_magnitude), interval):
        end = start + interval

        acc_mag_segment = linear_acc_magnitude[start:end]
        gyro_mag_segment = gyro_magnitude[start:end]

        if len(acc_mag_segment) == 0 or len(gyro_mag_segment) == 0:
            continue  

        acc_max = np.max(acc_mag_segment)
        gyro_max = np.max(gyro_mag_segment)
        acc_kurtosis = kurtosis(acc_mag_segment, fisher=True, bias=False) if len(acc_mag_segment) > 1 else 0
        gyro_kurtosis = kurtosis(gyro_mag_segment, fisher=True, bias=False) if len(gyro_mag_segment) > 1 else 0
        acc_skewness = skew(acc_mag_segment, bias=False) if len(acc_mag_segment) > 1 else 0
        gyro_skewness = skew(gyro_mag_segment, bias=False) if len(gyro_mag_segment) > 1 else 0

        segment_fraction = max(1, len(acc_mag_segment) // 5)
        post_lin_max = np.max(acc_mag_segment[-segment_fraction:]) if segment_fraction > 0 else 0
        post_gyro_max = np.max(gyro_mag_segment[-segment_fraction:]) if segment_fraction > 0 else 0

        features.append([
            acc_max,
            gyro_max,
            acc_kurtosis,
            gyro_kurtosis,
            acc_skewness,
            gyro_skewness,
            np.max(acc_mag_segment),  # linear acc max
            post_lin_max,
            post_gyro_max
        ])

    return np.array(features)


In [3]:
scaler = joblib.load("scaler.pkl")
knn_model = joblib.load("knn_model.pkl")
svm_model = joblib.load("svm_model.pkl")

In [4]:
acceleration_10hz = pd.read_csv("data/acceleration_10hz.csv")
angular_velocity_10hz = pd.read_csv("data/angular_velocity_10hz.csv")

acceleration_1hz = pd.read_csv("data/acceleration_1hz.csv")
angular_velocity_1hz = pd.read_csv("data/angular_velocity_1hz.csv")

In [5]:
nearest_indices_10hz = angular_velocity_10hz.index.map(
    lambda t: np.argmin(np.abs(acceleration_10hz.index - t))
)

angular_velocity_10hz['nearest_acc_timestamp'] = acceleration_10hz.index[nearest_indices_10hz]

combined_data_10hz = angular_velocity_10hz.join(acceleration_10hz, on='nearest_acc_timestamp', rsuffix='_acc')
combined_data_10hz.rename(columns={'X': 'X_gyro', 'Y': 'Y_gyro', 'Z': 'Z_gyro'}, inplace=True)
combined_data_10hz.drop('nearest_acc_timestamp', axis=1, inplace=True)
combined_data_10hz

,Timestamp,X_gyro,Y_gyro,Z_gyro,Timestamp_acc,X_acc,Y_acc,Z_acc
0,01-Feb-2025 18:26:58.005,-0.052534,0.076969,0.006109,01-Feb-2025 18:26:57.576,0.399832,9.890448,0.624887
1,01-Feb-2025 18:26:58.085,-0.117286,0.058643,0.023824,01-Feb-2025 18:26:57.656,0.402226,9.701305,1.498770
2,01-Feb-2025 18:26:58.165,-0.129503,0.070860,0.002443,01-Feb-2025 18:26:57.736,0.876278,9.876082,-0.028730
3,01-Feb-2025 18:26:58.245,-0.088575,0.009774,-0.016493,01-Feb-2025 18:26:57.816,0.423774,9.900024,0.926556
4,01-Feb-2025 18:26:58.325,0.098960,-0.006720,0.003054,01-Feb-2025 18:26:57.896,0.605733,9.950302,0.773327
...,...,...,...,...,...,...,...,...
1532,01-Feb-2025 18:29:00.633,-0.169210,0.030543,-0.039095,01-Feb-2025 18:29:00.204,0.715866,4.762067,9.119514
1533,01-Feb-2025 18:29:00.714,0.059254,-0.099571,-0.047037,01-Feb-2025 18:29:00.284,0.763750,4.702212,8.868123
1534,01-Feb-2025 18:29:00.794,-0.030543,0.014050,-0.028711,01-Feb-2025 18:29:00.364,0.828394,4.738125,9.050082
1535,01-Feb-2025 18:29:00.874,0.012217,-0.101404,-0.009163,01-Feb-2025 18:29:00.445,0.725443,4.824317,8.449138


In [6]:
nearest_indices_1hz = angular_velocity_1hz.index.map(
    lambda t: np.argmin(np.abs(acceleration_1hz.index - t))
)

angular_velocity_1hz['nearest_acc_timestamp'] = acceleration_1hz.index[nearest_indices_1hz]

combined_data_1hz = angular_velocity_1hz.join(acceleration_1hz, on='nearest_acc_timestamp', rsuffix='_acc')
combined_data_1hz.rename(columns={'X': 'X_gyro', 'Y': 'Y_gyro', 'Z': 'Z_gyro'}, inplace=True)
combined_data_1hz.drop('nearest_acc_timestamp', axis=1, inplace=True)
combined_data_1hz

,Timestamp,X_gyro,Y_gyro,Z_gyro,Timestamp_acc,X_acc,Y_acc,Z_acc
0,01-Feb-2025 18:29:47.768,0.092241,-0.022602,-0.013439,01-Feb-2025 18:29:47.647,0.433351,5.102044,8.595184
1,01-Feb-2025 18:29:47.928,-0.057421,-0.222966,0.100793,01-Feb-2025 18:29:47.807,0.311246,5.073313,8.566454
2,01-Feb-2025 18:29:48.088,-0.054367,0.021380,-0.003054,01-Feb-2025 18:29:47.967,0.672771,4.972757,8.772355
3,01-Feb-2025 18:29:48.248,-0.029322,0.000611,-0.002443,01-Feb-2025 18:29:48.127,0.550666,5.121197,8.360553
4,01-Feb-2025 18:29:48.408,0.081856,-0.053145,0.050702,01-Feb-2025 18:29:48.287,0.505177,4.711789,9.217676
...,...,...,...,...,...,...,...,...
1156,01-Feb-2025 18:32:51.862,-0.095295,0.109956,-0.089797,01-Feb-2025 18:32:09.794,1.261744,10.024523,-0.950498
1157,01-Feb-2025 18:32:52.022,-0.191812,0.061697,-0.065363,01-Feb-2025 18:32:09.814,1.252167,10.407595,-0.933739
1158,01-Feb-2025 18:32:52.183,-0.199753,-0.039706,-0.009163,01-Feb-2025 18:32:09.834,1.015141,10.685322,-0.960075
1159,01-Feb-2025 18:32:52.343,0.010996,-0.020769,-0.004887,01-Feb-2025 18:32:09.854,0.711078,11.207258,-1.127669


In [7]:
features_coletado_10hz= get_features(combined_data_10hz["X_acc"],combined_data_10hz["Y_acc"], combined_data_10hz["Z_acc"], combined_data_10hz["X_gyro"], combined_data_10hz["Y_gyro"], combined_data_10hz["Z_gyro"], 100)
features_coletado_10hz

array([[ 4.56004908,  0.82476336,  1.50476832,  0.4539739 ,  1.14501933,
         0.76074589,  4.56004908,  2.75302751,  0.57428017],
       [ 3.29440182,  2.25101726, -0.29447696,  1.352366  ,  0.4438491 ,
         1.53505479,  3.29440182,  3.29440182,  0.66073157],
       [ 3.22441872,  2.38202325, -0.29893966,  1.41129967,  0.51560843,
         1.58908056,  3.22441872,  3.22441872,  0.90540924],
       [ 3.49374906,  2.49580208, -0.15149045,  2.36737189,  0.57340313,
         1.86494607,  3.49374906,  2.35082734,  1.88553575],
       [ 3.35622888,  1.98245947,  0.08306556,  9.07025414,  0.92170301,
         2.9262252 ,  3.35622888,  2.54439833,  1.98245947],
       [ 3.23398988,  1.95039756,  1.32518928,  2.46942235,  1.0304836 ,
         1.88261766,  3.23398988,  2.42734154,  0.61468234],
       [ 2.54431124,  2.11648934, -0.54725646,  0.60500451,  0.4761294 ,
         1.36168322,  2.54431124,  2.40924121,  0.60173477],
       [ 3.29399454,  2.23660252,  0.45582758,  0.35574853,  0

In [8]:
features_coletado_1hz= get_features(combined_data_1hz["X_acc"],combined_data_1hz["Y_acc"], combined_data_1hz["Z_acc"], combined_data_1hz["X_gyro"], combined_data_1hz["Y_gyro"], combined_data_1hz["Z_gyro"], 100)
features_coletado_1hz

array([[11.99878698,  2.24463283,  5.24034628,  2.49922264,  2.51453776,
         1.79937898, 11.99878698,  2.87123709,  0.56163218],
       [ 2.97809177,  2.23110941, -0.33439811,  0.57772015,  0.67290662,
         1.42034464,  2.97809177,  2.93421518,  0.50299785],
       [ 3.41381863,  2.11743472,  1.5133966 ,  1.05433385,  1.22522441,
         1.51538849,  3.41381863,  2.62295083,  2.0887133 ],
       [ 3.48147398,  2.42805054,  1.54874151,  2.99357177,  1.32830479,
         1.96986551,  3.48147398,  3.48147398,  2.42805054],
       [ 3.0776983 ,  2.47900897,  0.84027107,  5.0861471 ,  0.98208024,
         2.29524332,  3.0776983 ,  3.0776983 ,  1.96249373],
       [ 2.7879082 ,  2.13667558,  0.05605316,  2.58067665,  0.88548511,
         1.88410394,  2.7879082 ,  2.70686994,  0.598809  ],
       [ 3.26884818,  2.40679583,  1.12148613,  1.37301726,  1.06109376,
         1.53943295,  3.26884818,  2.77117722,  0.62772966],
       [ 3.59823036,  2.14974257,  1.09269303,  0.78191047,  1

In [9]:
X_coletado_10hz = np.array(features_coletado_10hz)

scaler.fit(X_coletado_10hz)
X_coletado_10hz = scaler.transform(X_coletado_10hz)

predicoes_coletado_knn_10hz = knn_model.predict(X_coletado_10hz)
predicoes_coletado_svm_10hz = svm_model.predict(X_coletado_10hz)


In [10]:
X_coletado_1hz = np.array(features_coletado_1hz)

scaler.fit(X_coletado_1hz)
X_coletado_1hz = scaler.transform(X_coletado_1hz)

predicoes_coletado_knn_1hz = knn_model.predict(X_coletado_1hz)
predicoes_coletado_svm_1hz = svm_model.predict(X_coletado_1hz)

In [11]:
predicoes_coletado_knn_1hz

array([1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0])

In [12]:
predicoes_coletado_svm_1hz

array([1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0])